# Part 1 — ETL and ELT
## Q. 1 — Perform a Small ETL Flow
Create a clean DataFrame where:
price_gbp is numeric;
invalid prices are rejected;
only valid rows are loaded into clean_products.

In [4]:
import pandas as pd
 
source = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "price_gbp": ["25.50", "invalid", "12.00", "40.00"],
})

working_source = source.copy()

working_source["price_gbp"] = pd.to_numeric(working_source["price_gbp"], errors="coerce")

clean_products = working_source.dropna(subset=["price_gbp"]).copy()

print(f"Clean products:\r\n {clean_products}")

Clean products:
    product_id  price_gbp
0         101       25.5
2         103       12.0
3         104       40.0


## Q. 2 — Simulate a Small ELT Flow

In [9]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [201, 202, 203],
    "amount_gbp": ["45.50", "30.00", "bad"],
})

raw_orders = source.copy()

clean_orders = raw_orders.copy()

clean_orders["amount_gbp"] = pd.to_numeric(clean_orders["amount_gbp"], errors="coerce")

clean_orders = clean_orders.dropna(subset=["amount_gbp"]).copy()

print(f"Raw orders:\r\n {raw_orders}")
print(f"Clean orders:\r\n {clean_orders}")



Raw orders:
    order_id amount_gbp
0       201      45.50
1       202      30.00
2       203        bad
Clean orders:
    order_id  amount_gbp
0       201        45.5
1       202        30.0


## Q. 3 — Choose ETL or ELT
Requirement	                                 Choice	                         Reason

Remove invalid data before target	         ETL	                         Data is transformed and cleaned before loading into the target.
Keep raw data first	                         ELT	                         Raw data is loaded first and can be retained for future transformations.
Strong target SQL capability	             ELT	                         Transformation can happen in the target
Remove sensitive fields before loading	     ETL	                         Sensitive values are changed before the target receives them

## Q. 4 — MCQ: ETL
Which sequence represents ETL?
Answer:
C. Extract → Transform → Load 

# Part 2 — Batch, Streaming, and Raw Data
## Q. 5 — Process Three Independent Batches

In [12]:
batches = {
    "batch_1": [10, 20, 30],
    "batch_2": [40, 50],
    "batch_3": [60, 70, 80],
}

for batch_name, rows in batches.items():
    row_count = len(rows)
    batch_total = sum(rows)

    print("Batch name:", batch_name)
    print("Row count:", row_count)
    print("Batch total:", batch_total)
    print()
   

Batch name: batch_1
Row count: 3
Batch total: 60

Batch name: batch_2
Row count: 2
Batch total: 90

Batch name: batch_3
Row count: 3
Batch total: 210



# Q. 6 — Batch or Streaming?
Choose the better processing style.

1. Supplier sends one file every night. - Batch
2. Payment events arrive continuously and need a result within seconds. - Streaming
3. Finance produces one monthly report. - Batch
4. Website click events arrive throughout the day. - Streaming

# Q. 7 — Preserve Raw Input Before Cleaning
Rules for clean_orders:
city must not be missing;
amount must be positive;
trim spaces from city.
Confirm that raw_orders still has all original rows.

In [15]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [301, 302, 303],
    "city": [" London ", None, "Leeds"],
    "amount_gbp": [45.50, 20.00, -5.00],
})

raw_orders = source.copy()

clean_orders = raw_orders.copy()

clean_orders["city"] = clean_orders["city"].str.strip()

clean_orders = clean_orders[
    clean_orders["city"].notna()
    & (clean_orders["amount_gbp"] > 0)
].copy()

print(f"Raw orders:\r\n {raw_orders}")
print(f"Clean orders:\r\n {clean_orders}")





Raw orders:
    order_id      city  amount_gbp
0       301   London         45.5
1       302       NaN        20.0
2       303     Leeds        -5.0
Clean orders:
    order_id    city  amount_gbp
0       301  London        45.5


## Q. 8 — MCQ: Streaming
Which requirement is the strongest reason to use streaming?

Answer: A. Fraud events must be checked within seconds. 

# Part 3 — Staging, Transformation, and Analytics Layers
## Q. 9 — Place Work in the Correct Layer
Classify each activity as mainly staging, transformation, or analytics.

A. Rename source columns to standard names. - staging
B. Join orders to customers. - transf
C. Calculate daily revenue by city. - analytics
D. Convert an amount column from text to numeric. - staging or transf
E. Create a report-ready monthly sales table. - analytics

## Q. 10 — Build a Staging Output


In [16]:
import pandas as pd
 
source = pd.DataFrame({
    "OrderID": [401, 402, 403],
    "CustomerID": [1, 2, 1],
    "Amount": ["45.50", "30.00", "20.00"],
})

staging_orders = source.copy()

staging_orders = staging_orders.rename(columns={
    "OrderID": "order_id",
    "CustomerID": "customer_id",
    "Amount": "amount_gbp"
})

staging_orders["amount_gbp"] = pd.to_numeric(
    staging_orders["amount_gbp"], errors="coerce"
)

print(staging_orders)





   order_id  customer_id  amount_gbp
0       401            1        45.5
1       402            2        30.0
2       403            1        20.0


## Q. 11 — Build an Analytics Output

In [17]:
import pandas as pd
 
orders = pd.DataFrame({
    "order_id": [501, 502, 503, 504],
    "city": ["London", "Leeds", "London", "Manchester"],
    "amount_gbp": [45.50, 30.00, 62.25, 50.00],
    "status": ["complete", "complete", "complete", "cancelled"],
})

completed_orders = orders[orders["status"] == "complete"]

report = (
    completed_orders
    .groupby("city", as_index=False)["amount_gbp"]
    .sum()
    .rename(columns={"amount_gbp": "completed_revenue_gbp"})
)

print(report)


     city  completed_revenue_gbp
0   Leeds                  30.00
1  London                 107.75


## Q. 12 — MCQ: Staging
Which task is most suitable for a staging layer?

Answer: D. Rename and standardise source columns before deeper transformation.

# Part 4 — Dependencies, Retries, and Backfills
## Q. 13 — Put the Jobs in Order

Write the dependency chain:
load_source
    ↓
clean_orders
    ↓
build_daily_sales
    ↓
publish_report

## Q. 14 — Retry or Fix First?
Choose whether an automatic retry is useful.

Failure	                                    Retry useful?	Reason
Temporary network timeout	                Yes	            The connection may recover on the next attempt.
Python syntax error	                        No	            The code must be corrected before it can run successfully.
Database unavailable for 30 seconds	        Yes	            The database may become available again after a short delay.
Required column removed from source file	No	            The source schema must be corrected or the code updated to handle the change.

## Q. 15 — Plan a Backfill
The daily pipeline did not run on:
2026-08-05
2026-08-06
2026-08-07

Create a backfill plan.

Include:
processing order
duplicate check
validation after each date

Solution:

Confirm the source data exists for all three dates.
Check whether any partial output already exists.
Process 2026-08-05.
Validate counts and totals.
Process 2026-08-06.
Validate counts and totals.
Process 2026-08-07.
Validate counts and totals.
Confirm that rerunning a date will not duplicate data.


## Q. 16 — MCQ: Dependency
Why is a dependency needed between clean_orders and build_daily_sales?

Answer: B. To ensure the sales summary waits for clean data.

# Optional Practice
## Q. 17 — Build a Complete Independent ETL Script

Rules:
keep the raw copy unchanged;
reject missing city;
reject non-numeric amount;
use only completed orders in daily_sales;
aggregate revenue by city.

In [20]:
import pandas as pd
 
source = pd.DataFrame({
    "order_id": [601, 602, 603, 604],
    "city": ["London", None, "Leeds", "Manchester"],
    "amount_gbp": ["45.50", "30.00", "bad", "50.00"],
    "status": ["complete", "complete", "complete", "cancelled"],
})

raw_orders = source.copy()

clean_orders = raw_orders.copy()

clean_orders[ "amount_gbp"] = pd.to_numeric(clean_orders[ "amount_gbp"], errors="coerce")

clean_orders = clean_orders[clean_orders["city"].notna()].copy()

clean_orders = clean_orders.dropna(subset=["amount_gbp"]).copy()

completed_orders = clean_orders[clean_orders["status"] == "complete"].copy()

daily_sales = (
    completed_orders.groupby("city", as_index=False)["amount_gbp"].sum().rename(columns={ "amount_gbp": "completed_revenue_gbp"}))

print("Raw:")
print(raw_orders)
 
print("Clean:")
print(clean_orders)

print("\nDAILY SALES")
print(daily_sales)


Raw:
   order_id        city amount_gbp     status
0       601      London      45.50   complete
1       602         NaN      30.00   complete
2       603       Leeds        bad   complete
3       604  Manchester      50.00  cancelled
Clean:
   order_id        city  amount_gbp     status
0       601      London        45.5   complete
3       604  Manchester        50.0  cancelled

DAILY SALES
     city  completed_revenue_gbp
0  London                   45.5


Q. 18 — Find the Pipeline Design Problem
A team runs:

publish_report
build_daily_sales
clean_orders
load_source

in that order.

Answer:
load_source
    ↓
clean_orders
    ↓
build_daily_sales
    ↓
publish_report



## Q. 19 — Check Whether a Backfill Is Safe
A backfill script always uses:
'append' when writing results.

If 2026-08-05 is accidentally processed twice, the same rows may appear twice.
Explain why this is risky and what the team should check before rerunning.

Using 'append' is risky because rerunning the same date can insert duplicate rows, causing incorrect sales totals and reports.

Before rerunning a backfill, the team should check whether data for the date already exists, identify duplicate records using unique keys, inspect any partial output from previous runs, and confirm the write strategy.

The team should use a safe approach, such as replacing the affected date's data or performing an idempotent upsert. After processing, they should validate row counts and revenue totals against the source data.


## Q. 20 — Design a Four-Layer Batch Pipeline
Design an independent pipeline containing:
raw
staging
transformation
analytics

Use a daily order file as the source.

Answer:

Layer	            Input	            Main work	                            Output	        Validation
Raw	                Daily order file	Keep received data	                    Raw orders	    Input row count
Staging	            Raw orders	        Standardise columns and types	        Staged orders	Required columns
Transformation	    Staged orders	    Clean, join, apply business rules	    Trusted orders	Valid/rejected counts
Analytics	        Trusted orders	    Aggregate for reporting	                Daily sales	    Revenue reconciliation
